# Memory

## Overview
A model has no memory of its own: each call starts from whatever text you send. Agent memory is therefore an engineering problem with two parts:

| Kind | Lives in | Lasts | Strands tool |
|:--|:--|:--|:--|
| **Short-term** (the conversation) | `agent.messages` | One conversation, unless you persist it | `SlidingWindowConversationManager`, `FileSessionManager` |
| **Long-term** (facts about the user) | A vector store | Across sessions and restarts | [mem0](https://docs.mem0.ai) as a tool |

This notebook does both, with a **file-based Qdrant** vector store (no Docker, no server) and the embedding model from `config.yaml`. The agent also has a web-search tool (DuckDuckGo, needs internet) to show memory and tools working together.

The notebook uses a single agent with two custom tools (mem0_memory for store / retrieve / list, and websearch). The memory backend is mem0 on the Qdrant server from `config.yaml` (`qdrant.url`). Strands features covered: `FileSessionManager`, `SlidingWindowConversationManager` and tools with direct invocation. The model provider and model come from `config.yaml`.

## Architecture

<div style="text-align:center">
    <img src="images/memory_architecture.svg" width="85%" />
</div>

## What it does
* Shows why a conversation is not memory, and persists it with a session manager
* Stores, retrieves and lists long-term memories with mem0
* Lets the agent decide what to remember and what to recall
* Proves memory survives a brand-new agent (a new session)

## Setup and prerequisites

* The chat model and the embedding model from `config.yaml` available
* `pip install -r requirements.txt` (adds `mem0ai`, `ddgs`)

mem0 makes its own LLM calls to extract facts from text, using the same model as the agent. Extraction quality depends on the model, so check what was stored (the notebook lists it).

In [1]:
#!pip install -r requirements.txt

In [2]:
import os
os.environ["MEM0_TELEMETRY"] = "False"      # keep this notebook fully offline

import logging, warnings
import yaml
from strands import Agent, tool
from strands.models.ollama import OllamaModel

warnings.filterwarnings("ignore")
logging.getLogger("strands").setLevel(logging.WARNING)

cfg = yaml.safe_load(open("config.yaml"))

def make_model(**overrides):
    """One place to build the model from config.yaml."""
    params = dict(
        host=cfg["ollama"]["url"],
        model_id=cfg["ollama"]["models"][0],
        temperature=cfg["strands"]["temperature"],
        options={"num_ctx": cfg["strands"]["num_ctx"]},
    )
    params.update(overrides)
    return OllamaModel(**params)

model = make_model()
print("Model:", model.config["model_id"], "@", cfg["ollama"]["url"])

Model: gemma4:e2b-mlx @ http://localhost:11434


## Part 1: Short-term memory

### The conversation is the memory

Within one `Agent` object, follow-up questions work because `agent.messages` carries the history:

In [3]:
agent = Agent(model=model, system_prompt="You are a concise assistant.", callback_handler=None)
agent("My name is Ada and I live in Chicago.")
print(agent("What is my name and where do I live?"))
print("messages kept:", len(agent.messages))

Your name is Ada, and you live in Chicago.

messages kept: 4


A new `Agent` object starts empty. The history was in the object, not in the model:

In [4]:
fresh = Agent(model=model, system_prompt="You are a concise assistant.", callback_handler=None)
print(fresh("What is my name?"))

I do not know your name.



### Keeping the history bounded

A long conversation eventually overflows the context window. The default `SlidingWindowConversationManager` drops the oldest messages once the window is full and trims oversized tool results. Set the window explicitly when you need to control cost:

In [5]:
from strands.agent.conversation_manager import SlidingWindowConversationManager

short = Agent(model=model, callback_handler=None,
              conversation_manager=SlidingWindowConversationManager(window_size=6))
for i in range(6):
    short(f"Remember the number {i}. Reply with OK.")
print("messages kept (window_size=6):", len(short.messages))

messages kept (window_size=6): 6


### Persisting the conversation with a session manager

A **session manager** saves messages after every turn and reloads them when an agent starts with the same `session_id`. That gives you short-term memory that survives a restart. Here it writes JSON files under a local directory:

In [6]:
import shutil
from strands.session.file_session_manager import FileSessionManager

SESSION_DIR = "/tmp/week7_sessions"
shutil.rmtree(SESSION_DIR, ignore_errors=True)

def session_agent():
    return Agent(model=model, system_prompt="You are a concise assistant.", callback_handler=None,
                 session_manager=FileSessionManager(session_id="ada-chat", storage_dir=SESSION_DIR))

first = session_agent()
first("My favourite colour is teal. Please remember it.")
del first                                    # the first agent is gone

second = session_agent()                    # a new object, same session_id
print("messages restored:", len(second.messages))
print(second("What is my favourite colour?"))

messages restored: 2
Your favorite color is teal.



Sessions hold one conversation verbatim. They do not search, summarise or generalise. For *facts about a user* that should be recalled across many conversations, you want long-term memory.

## Part 2: Long-term memory with mem0

[mem0](https://docs.mem0.ai) turns text into short, deduplicated **facts** (an LLM does the extraction) and stores them in a vector database. Later you search by meaning.

### Configuration

Everything is local and driven by `config.yaml`:

| Piece | Setting | Value here |
|:--|:--|:--|
| Vector store | `qdrant.url`, `memory.collection` | Qdrant server at `http://localhost:6333` (start it first; collection visible in its dashboard) |
| Extraction LLM | chat model in `config.yaml` | same model as the agent |
| Embeddings | embedding model in `config.yaml` | dimensions set by `embed_dims` |

In [7]:
with warnings.catch_warnings():              # mem0 lazily imports typer, which warns about click 9 removals
    warnings.simplefilter("ignore", DeprecationWarning)
    import typer                             # noqa: F401  (cache it quietly)
    from mem0 import Memory

from qdrant_client import QdrantClient
QdrantClient(url=cfg["qdrant"]["url"]).delete_collection(cfg["memory"]["collection"])   # start with an empty memory (this collection only)

mem0_config = {
    "vector_store": {"provider": "qdrant", "config": {
        "url": cfg["qdrant"]["url"],
        "collection_name": cfg["memory"]["collection"],
        "embedding_model_dims": cfg["ollama"]["embed_dims"],
    }},
    "llm": {"provider": "ollama", "config": {
        "model": cfg["ollama"]["models"][0],
        "ollama_base_url": cfg["ollama"]["url"],
        "temperature": 0, "max_tokens": 2000,
    }},
    "embedder": {"provider": "ollama", "config": {
        "model": cfg["ollama"]["embed_model"],
        "ollama_base_url": cfg["ollama"]["url"],
        "embedding_dims": cfg["ollama"]["embed_dims"],
    }},
}
memory = Memory.from_config(mem0_config)
print("mem0 ready | Qdrant collection", cfg["memory"]["collection"], "@", cfg["qdrant"]["url"])
print("points stored:", memory.vector_store.client.count(cfg["memory"]["collection"]).count)

mem0 ready | Qdrant collection agent_memories @ http://localhost:6333
points stored: 0


### The memory tool

One tool with three actions, so the model can choose between them. The tool is **bound to one user** when it is built (`make_memory_tool(user_id)`). We do not let the model pass a `user_id`: small models forget it, and, more importantly, the model must never be the one deciding whose memories to read. Identity comes from your application, such as the logged-in user.

| Action | Description |
|:--|:--|
| `store` | Save information for a `user_id` |
| `retrieve` | Semantic search over that user's memories |
| `list` | Return everything stored for that user |

Each user's memories are kept apart by `user_id` inside the store.

In [8]:
def make_memory_tool(user_id: str):
    """Build a memory tool bound to ONE user. The application, not the model, decides whose memory this is."""

    @tool
    def mem0_memory(action: str, content: str = "", query: str = "") -> str:
        """Store, retrieve or list long-term memories about the current user.

        Args:
            action: One of 'store', 'retrieve' or 'list'.
            content: The information to remember (required for 'store').
            query: What to look for (required for 'retrieve').
        """
        if action == "store":
            result = memory.add(content, user_id=user_id)
            return ("Stored: " + "; ".join(r["memory"] for r in result["results"])) if result["results"] else "Nothing new to store."
        if action == "retrieve":
            hits = memory.search(query, filters={"user_id": user_id}, limit=5)["results"]
            return "\n".join(f"- {h['memory']} (score {h['score']:.2f})" for h in hits) or "No relevant memories."
        if action == "list":
            items = memory.get_all(filters={"user_id": user_id})["results"]
            return "\n".join(f"- {i['memory']}" for i in items) or "No memories yet."
        return f"Unknown action '{action}'. Use 'store', 'retrieve' or 'list'."

    return mem0_memory

### Web search tool

The agent also gets a web-search tool. This one needs the internet.

In [9]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException

@tool
def websearch(keywords: str, region: str = "us-en", max_results: int = 3) -> str:
    """Search the web to get updated information.

    Args:
        keywords: The search query keywords.
        region: The search region: wt-wt, us-en, uk-en, etc.
        max_results: The maximum number of results to return.
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return str(results) if results else "No results found."
    except RatelimitException:
        return "RatelimitException: Please try again after a short delay."
    except DDGSException as d:
        return f"DuckDuckGoSearchException: {d}"
    except Exception as e:
        return f"Exception: {e}"

### Define the system prompt

The system prompt is what makes the model *use* memory. A small local model needs the rules spelled out, including the user id to pass.

In [10]:
USER_ID = "ada"
mem0_memory = make_memory_tool(USER_ID)

SYSTEM_PROMPT = """You are a helpful personal assistant with long-term memory about the current user.

Rules:
- When the user tells you a fact or preference about themselves, call mem0_memory with action="store" and content=<the fact>.
- When the user asks about themselves or asks for a personalised recommendation, FIRST call mem0_memory with action="retrieve" and a query, then answer using the result.
- Use websearch only for current events or information you do not know.
- Be brief. If you have no stored information, say so."""

def new_agent():
    return Agent(model=model, system_prompt=SYSTEM_PROMPT, tools=[mem0_memory, websearch], callback_handler=None)

memory_agent = new_agent()

### Memory operations by direct call

Seed the memory by calling the tool directly (no model decision involved), then read it back. This is the same `agent.tool.<name>()` pattern from 7.1.

In [11]:
memory_agent.tool.mem0_memory(action="store", content="My name is Ada.")
memory_agent.tool.mem0_memory(action="store", content="I like to drink tea more than coffee.")
memory_agent.tool.mem0_memory(action="store", content="I am allergic to peanuts.")
print(memory_agent.tool.mem0_memory(action="list")["content"][0]["text"])
print("points in Qdrant:", memory.vector_store.client.count(cfg["memory"]["collection"]).count)

- User prefers drinking tea over coffee.
- User's name is Ada
- User has a peanut allergy.
points in Qdrant: 3


In [12]:
for q in ["What is the user's name?", "What does the user like to drink?", "food allergies"]:
    print(f"Q: {q}")
    print(memory_agent.tool.mem0_memory(action="retrieve", query=q)["content"][0]["text"], "\n")

Q: What is the user's name?
- User's name is Ada (score 0.34)
- User has a peanut allergy. (score 0.24)
- User prefers drinking tea over coffee. (score 0.22) 

Q: What does the user like to drink?
- User prefers drinking tea over coffee. (score 0.48)
- User's name is Ada (score 0.28)
- User has a peanut allergy. (score 0.28) 

Q: food allergies
- User has a peanut allergy. (score 0.47)
- User prefers drinking tea over coffee. (score 0.19)
- User's name is Ada (score 0.16) 



Retrieval is by **meaning**: the query `food allergies` found the peanut fact without sharing a keyword with it.

### The agent uses memory on its own

Now let the model decide. First it should **store** something new, then **recall** facts it was never told in this conversation.

In [13]:
print(memory_agent("Please remember this about me: I am vegetarian."))
print("--- stored so far ---")
print(memory_agent.tool.mem0_memory(action="list")["content"][0]["text"])

I have remembered that you are vegetarian.

--- stored so far ---
- User prefers drinking tea over coffee.
- User's name is Ada
- User has a peanut allergy.
- User explicitly states that they are vegetarian.


In [14]:
print(memory_agent("Suggest one snack for me. Check what you know about me first."))

Based on what I know, I know that you are vegetarian and you have a peanut allergy.

Given those facts, I suggest **fruit and vegetables**, such as sliced apples with some sunflower seed butter, or carrot sticks with hummus (ensure the hummus is peanut-free).



### Memory survives a new session

Drop the agent entirely, build a new one with an empty conversation, and ask. Anything it knows now came from the vector store.

In [15]:
del memory_agent
brand_new = new_agent()
print("history length:", len(brand_new.messages))
print(brand_new("What do you remember about my food and drink preferences?"))

history length: 0
I remember a few things about your preferences:

*   You prefer drinking tea over coffee.
*   You explicitly stated that you are vegetarian.
*   You have a peanut allergy.
*   Your name is Ada.



### Audit and delete

Memory about people needs governance. You must be able to see what is stored and delete it on request.

In [16]:
for item in memory.get_all(filters={"user_id": USER_ID})["results"]:
    print(item["id"][:8], "|", item["memory"])

37ec1291 | User prefers drinking tea over coffee.
7ffc5562 | User's name is Ada
8fbc3701 | User has a peanut allergy.
b3ef160f | User explicitly states that they are vegetarian.


In [17]:
memory.delete_all(user_id=USER_ID)
print("after delete_all:", memory.get_all(filters={"user_id": USER_ID})["results"])

after delete_all: []


## Interactive usage

[memory-agent-cli/memory_agent.py](memory-agent-cli/memory_agent.py) is the same agent as a terminal chat loop:

```bash
python memory_agent.py
```

Try: `Remember that I prefer tea over coffee`, then later `What do I prefer to drink?`.

## Summary

1. **Short-term** memory is the message history. Bound it with a conversation manager and persist it with a session manager.
2. **Long-term** memory is facts in a vector store. mem0 extracts the facts and searches by meaning. The store keeps users apart by `user_id`.
3. The model only uses memory if the **system prompt and tool docstring** tell it how. Bind the user id in the tool, not in the prompt. Seed and inspect with direct tool calls.
4. Plan for deletion: `delete_all` and the audit listing are part of the design, not an afterthought.